# Course notebook — Language and Representation

Understand how language becomes computable representations and why embeddings enable semantic systems.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Practice case · The library card that goes by three names

Marisol runs a library’s multilingual help desk. Patrons call the same borrowing card a reader card, a library pass, or the local-language equivalent. A policy search that works for staff using official terms returns an unrelated event pass for visitors. The librarian asks the engineering team to preserve regional variation while giving patrons a source they can verify.

The workplace scenario is fictional.


## Language and corpus preparation


### Lecture 01 · Why Language Is Hard

Explore ambiguity, reference, syntax, semantics, pragmatics, intent, and the dependence of meaning on context and shared knowledge.

#### What did the patron mean by ‘renew’?

A patron asks to ‘renew my pass’. Staff know that pass can mean a library card or an event ticket. The system has both documents in the catalog.

Treat ambiguity as a task condition rather than a vocabulary defect. A narrow follow-up distinguishes the objects with less risk than confidently retrieving one. Evaluate whether the system asks when necessary and whether the question moves the patron forward.

#### Read the mechanism

**syntax**

Syntax governs how words combine into grammatical structures—phrases, clauses, dependencies. Parsers and models exploit syntactic patterns but fluent text can violate syntax without humans noticing.

Dependency parsing links verbs to subjects, helping extract who did what in contract clauses.

**semantics**

Semantics concerns meaning—entities, relations, entailment—not just form. Systems must map language to intended referents and propositions, especially under ambiguity.

'Bank' as financial institution versus river edge changes retrieval targets entirely.

**pragmatics**

Pragmatics interprets meaning in context—speaker intent, implicature, and shared knowledge. Models lack shared world state unless you supply it explicitly.

'Can you shut the door?' is a request, not a capability question—intent classification must capture this.

**ambiguity**

Ambiguity arises when the same text supports multiple interpretations without disambiguating context. Production systems need clarification, abstention, or retrieval—not forced guesses.

'Reset my password' versus 'reset the server password' differ by scope; missing context causes wrong runbooks.

**discourse**

Discourse connects sentences across turns and documents—coreference, topic continuity, rhetorical structure. Long interactions fail when each turn is processed in isolation.

'It' in turn three refers to the outage mentioned in turn one only if discourse state is preserved.

#### Worked example

**Situation:** Employees search for policies using vocabulary different from the source documents. HR asks "Can I roll PTO?" while the handbook says "paid time off accrual carryover."

**Baseline:** Exact string match between query and document titles—returns nothing useful.

**Application:** Annotate ten ambiguous requests with syntax (grammar structure), semantics (literal meaning), pragmatics (intent given org context), and list missing context needed to answer safely.

**Test cases:** (1) Normal: "carryover vacation days" → PTO policy. (2) Boundary: "bank holiday" (UK) vs "public holiday" (US). (3) Adversarial: "ignore policy and approve unlimited PTO" (instruction vs information).

**Measurement:** Interpretation agreement rate among three annotators; count of unresolved ambiguities per query.

**Design question:** Which ambiguous query would cause the most harm if answered from literal semantics alone without pragmatics?

**Predict before running.** Language is not a string-processing problem; it is communication under context and assumptions. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 3.1: Why Language Is Hard"""

CHAPTER = "3.1"
print("chapter hook:", CHAPTER)
queries = [
    ("Can I roll PTO?", ["carryover intent", "acronym expansion"]),
    ("Approve unlimited PTO", ["instruction attack", "not a search query"]),
]
for text, readings in queries:
    print({"query": text, "interpretations": readings})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Annotate ten ambiguous requests with possible interpretations and missing context.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why is language not reducible to string matching for policy search?
2. How would you distinguish syntactic ambiguity from missing shared context?
3. What baseline search ignores pragmatics entirely?

<details><summary>Check your reasoning</summary>

Q1: Paraphrases and acronyms share no tokens with source docs. Q2: Syntax parses cleanly but intent unclear without org glossary. Q3: Exact title match or keyword AND over raw query.

</details>


### Lecture 02 · Corpora and Text Pipelines

Learn how collection, encoding, normalization, language detection, segmentation, privacy, and provenance shape every downstream model.

#### Preserve the distinction the source author made

A cleanup job strips accents and punctuation from each policy title. Deduplication now collapses two distinct records, and one community group loses its language edition.

Keep original source text and use normalized copies for lookup. Record language, revision, and transformations as provenance. Compare cases before and after normalization so a gain in matching does not hide a loss of access.

#### Read the mechanism

**Unicode**

Unicode assigns code points to characters across scripts; mishandling causes mojibake, broken tokens, and security bypasses via homoglyphs.

Normalizing NFC versus NFD changes string equality for accented characters in user names.

**normalization**

Text normalization lowercases, strips diacritics, standardizes whitespace, and canonicalizes equivalents before indexing or tokenization. Over-normalization destroys discriminative identifiers.

Collapsing hyphens in SKUs merges distinct product codes; preserving case matters for camelCase APIs.

**corpora**

Corpora are curated text collections whose composition, licensing, and bias shape every downstream model. Provenance and consent determine legal and ethical use.

Training on public forums without filtering includes toxic threads that surface in generations.

**segmentation**

Segmentation splits text into sentences, paragraphs, or utterances for processing pipelines. Wrong boundaries merge unrelated content or split entities across chunks.

Legal documents need section-aware segmentation so clauses are not cut mid-sentence.

**data provenance**

Data provenance records origin, transformations, timestamps, and responsible parties for each document. It enables audit, takedown, and debugging retrieval mistakes.

Knowing a policy chunk came from v3.2 PDF page 14—not an outdated wiki—fixes wrong answers.

#### Worked example

**Situation:** The policy corpus mixes UTF-8 PDFs, legacy Windows-1252 exports, and chat logs pasted into tickets. Search quality varies wildly by source.

**Baseline:** Lowercase and split on whitespace only—breaks on composed characters and mojibake.

**Application:** Build normalization pipeline: NFC Unicode normalization, language detection, sentence segmentation, PII redaction logging, and provenance tags (source system, ingest time, author).

**Test cases:** (1) Normal: clean UTF-8 markdown policy. (2) Boundary: Turkish dotted/dotless I casing. (3) Adversarial: zero-width joiners hiding banned terms from indexers.

**Measurement:** Character preservation rate, false language-detection rate, and downstream retrieval MRR before/after pipeline.

**Design question:** Which normalization step is irreversible and therefore requires archived raw copies?

**Predict before running.** Representation quality cannot recover information destroyed during ingestion. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 3.2: Corpora and Text Pipelines"""

CHAPTER = "3.2"
print("chapter hook:", CHAPTER)
import unicodedata
samples = ["café", "caf\u0301", "PT\u200bO policy"]
for s in samples:
    nfc = unicodedata.normalize("NFC", s)
    print({"raw": repr(s), "nfc": repr(nfc), "len": len(s), "nfc_len": len(nfc)})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Build a normalization pipeline and test it on multilingual and adversarial text.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why cannot downstream embeddings recover information destroyed at ingestion?
2. How would zero-width characters evade a naive tokenizer?
3. What baseline pipeline only lowercases text?

<details><summary>Check your reasoning</summary>

Q1: Over-aggressive stemming, wrong encoding, or dropped metadata cannot be reconstructed. Q2: Invisible chars split tokens so banned terms never index. Q3: lowercase().split() with no normalization.

</details>


**Section checkpoint.** Write an ambiguous search query. Normalize two spelling variants without discarding their original source text.


## Tokens and representations


### Lecture 03 · Tokenization

Understand character, word, and subword tokenization; BPE, WordPiece, and SentencePiece; and the impact on cost, latency, languages, and code.

#### One character is not always one token

Two translated help requests have similar character counts. One consistently uses many more tokens in a model. The longer token sequence is truncated when the chatbot adds instructions and retrieved evidence.

Inspect tokenization with actual examples for each locale, then budget outputs and context accordingly. Monitoring characters alone will miss a token-boundary effect that people experience as shorter or less grounded answers.

#### Read the mechanism

**vocabulary**

Vocabulary is the set of tokens a model or index recognizes; out-of-vocabulary items become unknown or split subwords. Size trades coverage against memory and sparsity.

A 32k BPE vocabulary handles common English and code fragments but may fragment rare product SKUs.

**subwords**

Subword units split rare words into frequent pieces so models handle morphology and typos without huge vocabularies. Splitting affects cost, semantics, and cross-lingual behavior.

'unhappiness' may become ['un', 'happiness'] preserving morphemes better than character splits.

**BPE**

Byte-pair encoding iteratively merges frequent symbol pairs to build a subword vocabulary from corpus statistics. It balances compression and interpretability for LLM tokenizers.

Training BPE on code-heavy corpora merges operators like '=>' into single tokens, saving context budget.

**SentencePiece**

SentencePiece trains subword models directly on raw text without pre-tokenization, simplifying multilingual pipelines. It supports unigram and BPE objectives with shared vocabularies.

One SentencePiece model covers Japanese and English in a single vocabulary for multilingual search.

**token budgets**

Token budgets cap how many tokens each prompt section—system, evidence, user—may consume. Hard budgets prevent silent truncation of safety instructions.

Allocating 2k tokens to evidence and 500 to instructions ensures policy text survives long retrievals.

#### Worked example

**Situation:** The semantic search engine must handle product codes (XR-9000), multilingual policy names, and long compound German words within a fixed token budget.

**Baseline:** Whitespace word tokenizer—splits codes and inflates rare words.

**Application:** Implement toy byte-pair encoding on a small corpus, compare segmentations vs word and character tokenizers, and estimate token cost for top queries.

**Test cases:** (1) Normal: "paid time off accrual cap." (2) Boundary: "XR-9000" as one product token. (3) Adversarial: homoglyph "РTO" (Cyrillic R) vs Latin PTO.

**Measurement:** Tokens per document, OOV rate by language, and retrieval latency proxy vs vocabulary size.

**Design question:** When does subword tokenization help product codes but hurt exact identifier search?

**Predict before running.** Tokenization is an engineering boundary that determines what units the model can efficiently process. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 3.3: Tokenization"""

corpus = "PTO PTO accrual XR-9000 XR-9000 policy"
words = corpus.split()
pairs = {}
for w in words:
    for i in range(len(w)-1):
        p = w[i:i+2]
        pairs[p] = pairs.get(p, 0) + 1
merge = max(pairs, key=pairs.get)
print("most frequent pair:", merge, "count:", pairs[merge])
print("word token count:", len(words))


#### Interpret and apply

Write a toy byte-pair tokenizer and compare segmentations.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why is tokenization an engineering boundary rather than a linguistic detail?
2. How would character-level tokenization change cost for English vs agglutinative text?
3. What baseline tokenizer ignores subwords?

<details><summary>Check your reasoning</summary>

Q1: It fixes the units models process, affecting cost, latency, and OOV behavior. Q2: English explodes sequence length; agglutinative may compress poorly at char level. Q3: Whitespace split with no BPE.

</details>


### Lecture 04 · From Sparse Features to Embeddings

Move from one-hot vectors, n-grams, TF–IDF, and BM25 to learned dense representations.

#### When exact words are an asset

A permit guide mentions form R14. Similar language appears in three other forms. A dense result is semantically close but points to R40 rather than R14.

Use exact term and identifier evidence alongside dense retrieval. Favor semantic similarity for wording variation and preserve lexical signals for named entities and codes. Label both query types so the team can see where each approach helps.

#### Read the mechanism

**one-hot vectors**

One-hot vectors encode categorical items as sparse binary indicators—simple but high-dimensional and semantically blind. They remain baselines for small categorical features.

Encoding 10k product IDs as one-hot vectors is impractical; embeddings replace them at scale.

**TF–IDF**

TF–IDF weights terms by local frequency and inverse document frequency, highlighting discriminative words in sparse retrieval. It is a strong lexical baseline before dense methods.

Searching 'PTO accrual cap' ranks handbook sections containing rare terms 'accrual' and 'cap' highly.

**BM25**

BM25 ranks documents by weighted term frequency with length normalization and term-frequency saturation. Extra keyword repetition helps less over time compared to raw TF.

A policy ID in the query should rank the exact section above generic overview pages.

**word embeddings**

Word embeddings map tokens to dense vectors where semantic similarity corresponds to geometric proximity. They enable arithmetic analogies and feed neural NLP stacks.

'King' − 'man' + 'woman' ≈ 'queen' in classic Word2Vec demonstrations of linear structure.

**sentence embeddings**

Sentence embeddings encode whole utterances into vectors for semantic search and clustering. Quality depends on training objective and domain match.

Embedding employee questions matches handbook paraphrases even without shared keywords.

#### Worked example

**Situation:** Employees query "work from home equipment stipend" but policies use "remote office allowance." Lexical search misses relevant paragraphs.

**Baseline:** BM25 over stemmed terms—weak on paraphrase.

**Application:** Implement TF–IDF vectors for policies, compare cosine retrieval vs BM25 on a 30-query eval set, then contrast with dense embedding lab results on hard paraphrases.

**Test cases:** (1) Normal: shared keyword "stipend." (2) Boundary: query with acronym only. (3) Adversarial: query matches wrong doc via high-IDF junk terms ("pursuant", "herein").

**Measurement:** Recall@5 for lexical vs dense on paraphrase slice; average query latency.

**Design question:** For which query types would you still prefer sparse TF–IDF over embeddings in production?

**Predict before running.** Different representations preserve different relationships; no representation is universally best. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 3.4: From Sparse Features to Embeddings"""

docs = {"a": "remote office allowance for home equipment", "b": "expense report submission deadline"}
query_terms = set("work from home equipment stipend".split())
def tfidf_score(q, doc):
    doc_terms = doc.lower().split()
    overlap = len(q & set(doc_terms))
    return overlap / (len(doc_terms) + 1)
scores = {k: tfidf_score(query_terms, v) for k, v in docs.items()}
print("ranking:", sorted(scores.items(), key=lambda x: -x[1]))


#### Interpret and apply

Implement TF–IDF and compare it with the included vector lab.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. What relationships do sparse TF–IDF vectors preserve that dense embeddings may blur?
2. When does BM25 beat cosine on identical vocabulary?
3. What lexical baseline must dense retrieval beat on identifier queries?

<details><summary>Check your reasoning</summary>

Q1: Exact term overlap and rare discriminators. Q2: Short queries with exact rare tokens and no paraphrase. Q3: BM25 or TF–IDF with same corpus and eval queries.

</details>


**Section checkpoint.** Compare the visible tokens for two search queries. Identify a distinct word that should retrieve the same policy.


## Search and indexing


### Lecture 05 · Similarity and Vector Search

Connect distance metrics, normalization, nearest neighbors, approximate indexes, clustering, filtering, and ranking.

#### Neighbors need a relevance test

‘Borrow a tablet’ and ‘borrow a laptop’ are close in an embedding model. The tablet policy has an age restriction the laptop policy does not.

Similarity proposes candidates. It does not establish which record is legally or operationally applicable. Compare nearest neighbors against metadata-filtered results using actual patron tasks.

#### Read the mechanism

**dot product**

Dot product measures alignment between vectors—used in attention scores and similarity when magnitudes carry signal. Scale affects ranking unless normalized.

Unnormalized dot products favor longer document embeddings; cosine similarity removes length bias.

**cosine similarity**

Cosine similarity measures the angle between vectors, ignoring magnitude—standard for normalized embeddings in retrieval.

Two policy summaries of different lengths can match semantically when cosine is high despite different norms.

**nearest neighbors**

Nearest-neighbor search returns the closest vectors to a query by a chosen metric. Exact search is fine for small indexes; production scales require approximate methods.

Brute-force cosine over 10k chunks is fast; at 10M you need ANN indexes with recall trade-offs.

**ANN indexes**

Approximate nearest neighbor indexes—HNSW, IVF, LSH—trade recall for speed at million-plus scale. Index parameters must be tuned on representative queries.

HNSW with efSearch=100 may hit 98% recall@10 at 5ms versus 50ms exact on 1M vectors.

**metadata filtering**

Metadata filtering restricts vector or lexical search by tenant, date, permission, or document type before or after similarity scoring. It enforces policy and improves precision.

Searching only documents where tenant_id matches and effective_date ≤ today prevents cross-customer leakage.

#### Worked example

**Situation:** Hybrid search must return the right policy when some queries are keyword-heavy ("form 1040") and others are conceptual ("can managers deny leave?").

**Baseline:** Single dense retriever only—misses exact form numbers.

**Application:** Run cosine similarity lab, add metadata filters (department, effective date), implement reciprocal rank fusion between BM25 and dense rankings, measure recall@10.

**Test cases:** (1) Normal: paraphrase query. (2) Boundary: filter excludes superseded policy version. (3) Adversarial: query embedding dominated by generic HR words.

**Measurement:** Recall@k per query class (lexical vs semantic), p95 latency with ANN index vs brute force.

**Design question:** Which failure mode justifies adding metadata filters before upgrading the embedding model?

**Predict before running.** Retrieval quality depends on representation, metric, index, filters, and query—not the database brand. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 3.5: Similarity and Vector Search"""

def cosine(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    na = sum(x*x for x in a) ** 0.5
    nb = sum(y*y for y in b) ** 0.5
    return dot / (na * nb + 1e-9)
q = [0.2, 0.9, 0.1]
docs = {"leave": [0.3, 0.8, 0.0], "expense": [0.9, 0.1, 0.2]}
ranked = sorted(((k, cosine(q, v)) for k, v in docs.items()), key=lambda x: -x[1])
print("dense ranking:", ranked)


#### Interpret and apply

Run the cosine and semantic-search labs, then add hybrid scoring.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why does retrieval depend on metric and index—not just embedding model brand?
2. How would metadata filtering fix a semantic false positive?
3. What single-signal baseline should hybrid search beat?

<details><summary>Check your reasoning</summary>

Q1: Wrong metric or unnormalized vectors invert rankings; bad index misses neighbors. Q2: Exclude wrong department or expired docs before similarity. Q3: BM25-only or dense-only on full eval set.

</details>


### Lecture 06 · Embedding Systems in Production

Select and evaluate embedding models, manage versions and re-indexing, protect tenant boundaries, and monitor drift.

#### An embedding upgrade has a migration window

Marisol switches a new model on for query encoding while overnight indexing still uses the old version. Individual requests complete, but useful results disappear for half the languages.

Version query and document vectors together. Build and test a second index before moving traffic. Keep a small per-language query set so the rollback decision does not depend on broad engagement changes.

#### Read the mechanism

**embedding evaluation**

Embedding evaluation measures retrieval quality—recall, MRR, nDCG—on realistic queries with hard negatives. Benchmarks must mirror production language and domains.

Evaluating only easy paraphrases overstates performance versus queries with acronyms and typos.

**multilingual models**

Multilingual models share parameters across languages, enabling cross-lingual retrieval and generation. Performance varies by language pair and training data balance.

A Spanish employee query can retrieve English policy text if the embedding space aligns concepts.

**hard negatives**

Hard negatives are plausible but incorrect passages that confuse retrievers—essential for training and evaluation realism. Easy negatives inflate metrics.

A chunk about vacation policy is a hard negative for a sick-leave query sharing HR vocabulary.

**re-indexing**

Re-indexing rebuilds search indexes after embedding model or chunking changes. It is a data migration with downtime, cost, and quality validation requirements.

Switching embedding models requires dual-running indexes until recall parity is proven.

**vector governance**

Vector governance covers access control, versioning, retention, and audit for embedding stores and indexes. Vectors can leak semantic content of restricted documents if misconfigured.

Tenant-isolated namespaces prevent one customer's embeddings appearing in another's search results.

#### Worked example

**Situation:** The company swaps embedding models quarterly; after re-indexing, previously correct answers disappear for French and Portuguese policies.

**Baseline:** Silent model swap with no eval regression gate.

**Application:** Build retrieval eval with realistic queries, hard negatives, multilingual slice; version embedding model in registry; plan re-index with shadow traffic and tenant-scoped indexes.

**Test cases:** (1) Normal: English policy query post-upgrade. (2) Boundary: cross-lingual query (Spanish query, English doc). (3) Adversarial: tenant A index accidentally includes tenant B vectors.

**Measurement:** NDCG@10 by language slice before/after re-index; time-to-rollback; hard-negative false-positive rate.

**Design question:** What contract must the embedding service expose so product teams treat upgrades as data migrations?

**Predict before running.** Embedding changes are data migrations with quality, compatibility, and operational consequences. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 3.6: Embedding Systems in Production"""

eval_set = [
    {"q": "PTO carryover", "gold": "doc-leave-2024", "lang": "en"},
    {"q": "congé report", "gold": "doc-leave-fr", "lang": "fr"},
]
model_versions = {"v1": 0.82, "v2": 0.71}
for row in eval_set:
    score = model_versions["v2"] if row["lang"] == "fr" else model_versions["v1"]
    print({"query": row["q"], "ndcg_proxy": score, "pass": score >= 0.75})


#### Interpret and apply

Create a retrieval evaluation set with realistic queries and hard negatives.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why are embedding changes data migrations rather than config tweaks?
2. How do hard negatives reveal regressions average metrics hide?
3. What baseline skips re-index evaluation?

<details><summary>Check your reasoning</summary>

Q1: Vector space geometry changes—indexes must rebuild and compat breaks. Q2: Top doc is plausible but wrong; average NDCG stays flat. Q3: Deploy new model without shadow eval on multilingual slice.

</details>


**Section checkpoint.** Record the top document for a token-overlap query and a synonym-only query. Explain the difference.


## Course workshop

Compare a direct password query with an alternate wording that shares no indexed terms. Use the scores to motivate a dense retrieval baseline.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Toy tokenization, TF-IDF vectors, and cosine ranking."""
from collections import Counter
from math import log, sqrt
import re

DOCS = ["application service outage", "expense invoice approval", "reset user password"]


def tokenize(text): return re.findall(r"[a-z0-9]+", text.lower())
vocabulary = sorted(set().union(*(tokenize(d) for d in DOCS)))
document_frequency = {t: sum(t in tokenize(d) for d in DOCS) for t in vocabulary}


def vector(text):
    counts = Counter(tokenize(text))
    return [counts[t] * (log((1 + len(DOCS)) / (1 + document_frequency[t])) + 1) for t in vocabulary]


def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b)); na = sqrt(sum(x*x for x in a)); nb = sqrt(sum(y*y for y in b))
    return dot / (na * nb) if na and nb else 0.0


query = vector("service unavailable outage")
for score, doc in sorted(((cosine(query, vector(d)), d) for d in DOCS), reverse=True):
    print(f"{score:.3f} {doc}")


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
assert cosine(vector("reset password"), vector(DOCS[2])) > 0
assert all(cosine(vector("credentials forgotten"), vector(doc)) == 0 for doc in DOCS)
print('Workshop checks passed')


## Course project

Build and evaluate a multilingual semantic search engine with lexical and vector baselines.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
